In [16]:
import httpx
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.abspath('./src'))
from db_functions import DotaDB

In [2]:
db = DotaDB(local=True)

In [4]:
query = 'SELECT match_id, hero_id, kills, deaths, assists FROM player_match_stats'
results = db.query_select_to_df(query, columns=['match_id', 'hero_id', 'kills', 'deaths', 'assists'])

In [6]:
db.set_local_or_remote(local=False, schema='public')

In [11]:
query = '''
    UPDATE match_players 
    SET kills = %s, deaths = %s, assists = %s
    WHERE match_id = %s AND "heroId" = %s 
'''
rows = list(results[['kills', 'deaths', 'assists', 'match_id', 'hero_id']].itertuples(index=False, name=None))
db.query_executemany(query, params=rows)

In [17]:
db = DotaDB(schema='kaggle')

In [18]:
query = 'SELECT radiant_score, dire_score, match_id FROM main_metadata'
results = db.query_select_to_df(query, columns=['radiant_score', 'dire_score', 'id']) 

In [19]:
db.set_schema('public')

In [24]:
query = '''
    UPDATE match_details
    SET radiant_score = %s, dire_score = %s
    WHERE id = %s
'''
valid_data = results[~((results['radiant_score'] == 0) & (results['dire_score'] == 0))]
rows = list(valid_data.itertuples(index=False, name=None))
db.query_executemany(query, params=rows)

In [ ]:
results[~((results['radiant_score'] == 0) & (results['dire_score'] == 0))] # Filter matches where rad_score and dire_score are both not 0


,radiant_score,dire_score,id
1557,24,18,2247016304
1558,35,23,2247054693
1559,33,14,2247063943
1560,28,10,2247097496
1561,17,23,2247125533
...,...,...,...
201151,25,34,8702486501
201152,54,24,8702504924
201153,34,19,8702512244
201154,38,40,8702524545
